In [ ]:
from typing import TypedDict

import matplotlib as mpl
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import seaborn as sns
from pingouin import mwu
from statsmodels.stats.multitest import multipletests

In [ ]:
mpl.rcParams["font.family"] = "DejaVu Sans"

In [ ]:
class PlotConfig(TypedDict):
    title: str
    xlabel: str


def format_pval(p: float) -> str:
    """Return Nature-style formatted p-values."""
    if p < 0.001:
        return "P < 0.001"
    elif p < 0.01:
        return f"P = {p:.3f}".replace("0.", ".")
    elif p < 0.05:
        return f"P = {p:.3f}".replace("0.", ".")
    else:
        return "ns"


def generate_violin_panels(
    results_df: pd.DataFrame,
    statistics_df: pd.DataFrame,
    plot_config: dict[str, PlotConfig],
    save_path: str,
    cut: float | list[float] = 0,
):
    """
    Generate multi-panel violin plots with Mann-Whitney U tests
    in Nature journal style, arranged in a custom grid:
        A D
        B E
        C

    Adds a single shared legend at the bottom with the group colors.
    """
    group_column = "Interviewer"

    sns.set_theme(style="whitegrid")
    sns.set_context("paper", font_scale=1.2)

    # --- Consistent group order & palette across all panels ---
    # Use the categorical order if present; otherwise, use sorted unique values.
    groups = list(reversed(sorted(results_df[group_column].dropna().unique().tolist())))

    palette_list = sns.color_palette("deep", n_colors=len(groups))
    palette = dict(zip(groups, palette_list))

    variables = list(plot_config.keys())
    n_panels = len(variables)

    # Define custom grid (3 rows x 2 cols)
    fig, axes = plt.subplots(3, 2, figsize=(10, 10), sharey=False)
    axes = axes.flatten()

    # Map panel positions manually (A-E)
    # positions correspond to: A=0, D=1, B=2, E=3, C=4
    grid_order = [0, 2, 4, 1, 3, 5]

    # If fewer than 5 panels, slice accordingly
    n_used = min(n_panels, len(grid_order))
    plot_axes = [axes[i] for i in grid_order[:n_used]]

    # Hide unused axes
    for j in range(len(plot_axes), len(axes)):
        axes[j].set_visible(False)

    if not isinstance(cut, list):
        cut = [cut] * n_used

    for i, (ax, var, c) in enumerate(zip(plot_axes, variables, cut)):
        cfg = plot_config[var]

        if len(groups) >= 2:
            p = statistics_df.loc[cfg["title"]]["p-val"]
            p_text = format_pval(p)
        else:
            p_text = "n/a"

        if i in (3, 4, 5):
            # Hist panels
            # Combine data range across all groups
            all_values = results_df[var].dropna()
            min_val, max_val = all_values.min(), all_values.max()

            # Define shared bins based on global range
            if i in (4, 5):
                bins = np.linspace(1, 5, 30)
            else:
                bins = np.linspace(min_val, max_val, 30)

            # Plot each group using the same bins and consistent colors
            for grp in groups:
                subset = results_df[results_df[group_column] == grp]
                ax.hist(
                    subset[var],
                    bins=bins,
                    alpha=0.6,
                    label=grp,
                    color=palette[grp],
                    edgecolor="black",
                    linewidth=0.8,
                )
                ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

            # Overlay median lines
            for grp in groups:
                med = results_df[results_df[group_column] == grp][var].median()
                ax.axvline(
                    med,
                    color=palette[grp],
                    linestyle="--",
                    linewidth=1.2,
                    alpha=0.9,
                    zorder=3,
                )

            ax.set_ylabel("Count")

        else:
            # Main violin + box + swarm with consistent palette
            sns.violinplot(
                data=results_df,
                x=var,
                y=group_column,
                ax=ax,
                hue=group_column,
                palette=palette,
                inner="box",
                inner_kws=dict(box_width=10, whis_width=2),
                linewidth=1.2,
                alpha=0.9,
                cut=c,
            )
            sns.swarmplot(
                data=results_df,
                x=var,
                y=group_column,
                ax=ax,
                color="black",
                alpha=0.6,
            )
            ax.set_ylabel("")

        # Panel label (A, B, C, D, E...)
        ax.text(
            -0.15,
            1.05,
            chr(65 + i),
            transform=ax.transAxes,
            fontsize=12,
            fontweight="bold",
            va="center",
            ha="center",
        )

        # Titles and labels
        ax.set_title(cfg["title"], fontsize=12, fontweight="bold", pad=8)
        ax.set_xlabel(cfg["xlabel"], fontsize=10)

        # Remove any per-axes legends; we'll add a shared one at the bottom
        leg = ax.get_legend()
        if leg is not None:
            leg.remove()

        # p-value annotation
        ax.text(
            0.98,
            0.5,
            p_text,
            transform=ax.transAxes,
            fontsize=10,
            ha="right",
            va="center",
            color="black",
            bbox=dict(facecolor="white", alpha=0.8, edgecolor="none", pad=2),
        )

        sns.despine(ax=ax, top=True, right=True)
        ax.set_facecolor("white")

    # --- Shared legend at the bottom ---
    handles = [
        mpatches.Patch(facecolor=palette[g], edgecolor="black", label=g)
        for g in groups
    ]
    # Make space at the bottom for the legend, then add it centered
    fig.tight_layout(h_pad=1.5, w_pad=1.5)
    fig.subplots_adjust(bottom=0.12)
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=len(groups),
        frameon=False,
        title=group_column,
        borderaxespad=0.5,
    )

    plt.savefig(save_path, dpi=600, bbox_inches="tight")
    plt.show()


In [ ]:
results_df = pd.read_csv("agg.csv")

In [ ]:
results_df_ai = results_df[results_df["Interviewer"] == "AI"]
results_df_human = results_df[results_df["Interviewer"] == "Human"]

# 1 - Users' Engagement

# Summary Statistics

In [ ]:
results_df.groupby("Interviewer")["question_count"].sum()

In [ ]:
# Group by 'Interviewer' and calculate summary statistics
summary_table = (
    results_df.groupby("Interviewer")[
        ["avg_response_length", "avg_response_duration_by_interviewer(sec)"]
    ]
    .agg(["mean", "median", "std", "min", "max"])
    .round(0)
)
summary_table.reset_index(inplace=True)
summary_table


In [ ]:
# Group by 'Interviewer' and calculate summary statistics
summary_table2 = (
    results_df.groupby("Interviewer")[["total_character_by_user", "question_count"]]
    .agg(["mean", "median", "std", "min", "max"])
    .round(1)
)
summary_table2.reset_index(inplace=True)
summary_table2


# Non-parametric Mann-Whitney U Test for Comparison

### Avg Response Duration

In [ ]:
res_dur_ai = results_df_ai["avg_response_duration_by_user(sec)"]
res_dur_human = results_df_human["avg_response_duration_by_user(sec)"]

mwu_res_dur = mwu(res_dur_ai, res_dur_human)
print(mwu_res_dur)

### Avg Response length

In [ ]:
res_length_ai = results_df_ai["avg_response_length"]
res_length_human = results_df_human["avg_response_length"]

mwu_res_length = mwu(res_length_ai, res_length_human)
print(mwu_res_length)

### Total Interview length by User 

In [ ]:
res_total_ai = results_df_ai["total_character_by_user"]
res_total_human = results_df_human["total_character_by_user"]

mwu_res_total = mwu(res_total_ai, res_total_human)
print(mwu_res_total)

### Question Count

In [ ]:
res_quest_ai = results_df_ai["question_count"]
res_quest_human = results_df_human["question_count"]

mwu_res_question = mwu(res_quest_ai, res_quest_human)
print(mwu_res_question)

### Question Formulation Time

In [ ]:
question_formulation_time_ai = results_df_ai["avg_response_duration_by_interviewer(sec)"]
question_formulation_time_human = results_df_human["avg_response_duration_by_interviewer(sec)"]

mwu_res_formulation = mwu(question_formulation_time_ai, question_formulation_time_human)
print(mwu_res_formulation)

**Conclusion**: There is statistically significant difference between AI and human interviews in terms of average response duration, total interview length and question count. 

# 2- Relevance and Specificity

In [ ]:
grouped_df = pd.read_csv("relevance_and_specificity_scores.csv")

## Summary Statistics

In [ ]:
# Group by 'Interviewer' and calculate summary statistics
summary_table3 = (
    grouped_df.groupby("interviewer")[["relevance", "specificity"]]
    .agg(["mean", "median", "std", "min", "max"])
    .round(1)
)
summary_table3.reset_index(inplace=True)
summary_table3

In [ ]:
results_df = results_df.merge(
    grouped_df.astype({"conversation_id": int}).rename(
        columns={"interviewer": "Interviewer"}, inplace=False
    ),
    on=["conversation_id", "Interviewer"],
)

## Non-parametric Mann-Whitney U Test

In [ ]:
grouped_df_ai = grouped_df[grouped_df["interviewer"] == "AI"]
grouped_df_human = grouped_df[grouped_df["interviewer"] == "Human"]

In [ ]:
relevance_ai = grouped_df_ai["relevance"]
relevance_human = grouped_df_human["relevance"]

mwu_relevance = mwu(relevance_ai, relevance_human)
print(mwu_relevance)

In [ ]:
specificity_ai = grouped_df_ai["specificity"]
specificity_human = grouped_df_human["specificity"]

mwu_specificity = mwu(specificity_ai, specificity_human)
print(mwu_specificity)

In [ ]:
stat_table = pd.concat(
    [
        mwu_res_length,
        mwu_res_total,
        mwu_res_question,
        mwu_res_formulation,
        mwu_relevance,
        mwu_specificity,
    ]
).drop("alternative", axis=1)

stat_table["p-val"] = multipletests(stat_table["p-val"].values, method="bonferroni")[1]

stat_table.index = [
    "Response Lengths",
    "Interview Length",
    "Questions per Interview",
    "Question Formulation Time",
    "Mean Specificity",
    "Mean Relevance",
]

print(
    stat_table.style.format(
        {"U-val": "{:.1f}", "p-val": "{:.3f}", "RBC": "{:.3f}", "CLES": "{:.3f}"}
    ).to_latex()
)

## Plot

In [ ]:
# Revised plot configuration
plot_config = {
    "avg_response_length": {
        "title": "Response Lengths",
        "xlabel": "Mean Number of Characters per Question Answered\n(aggregated by interview)",
    },
    "total_character_by_user": {
        "title": "Interview Length",
        "xlabel": "Total Number of Characters in Answers",
    },
    "question_count": {
        "title": "Questions per Interview",
        "xlabel": "Number of Questions Answered",
    },
    "avg_response_duration_by_interviewer(sec)": {
        "title": "Question Formulation Time",
        "xlabel": "Seconds",
    },
    "specificity": {
        "title": "Mean Specificity",
        "xlabel": "Mean Specificity Score per Interview",
    },
    "relevance": {
        "title": "Mean Relevance",
        "xlabel": "Mean Relevance Score per Interview",
    },
}

generate_violin_panels(results_df, stat_table, plot_config, "plots.png")